# RSNA 2018 — Execução da frente do Erik no Colab

Este notebook roda a parte pesada de **modelagem, ablação, avaliação e detecção**.
Os CSVs derivados ficam no Google Drive; o GitHub contém o código oficial.

In [ ]:
BRANCH = "Erik"  # após o merge desta frente, pode trocar para "main"
!rm -rf ap1-rsna-pneumonia-baseline
!git clone -q -b {BRANCH} https://github.com/erik-hp/ap1-rsna-pneumonia-baseline.git
%cd ap1-rsna-pneumonia-baseline
!pip -q install -r requirements.txt
!pip -q install -e .
!pytest -q

## 1. Montar o Drive

A pasta esperada é `MyDrive/RSNA_TP1/data/processed/`.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/RSNA_TP1")
PROCESSED = BASE / "data" / "processed"
OUT = BASE / "erik"
OUT.mkdir(parents=True, exist_ok=True)
FEATURES_ALL = PROCESSED / "features_all.csv"
FEATURES_REGIONS = PROCESSED / "features_regions.csv"
GROUND_TRUTH = PROCESSED / "ground_truth_boxes.csv"
IMAGE_MANIFEST = PROCESSED / "image_manifest.csv"
for p in [FEATURES_ALL, FEATURES_REGIONS, GROUND_TRUTH, IMAGE_MANIFEST]:
    print(("OK      " if p.exists() else "FALTANDO "), p)

## 2. Validar a matriz de classificação

In [ ]:
!python scripts/validate_features.py --features "{FEATURES_ALL}" --auto-ablation

## 3. Experimento principal — classificação por radiografia

Roda Dummy, SVM linear/RBF, Random Forest e HistGradientBoosting com nested CV 5×3 e todas as combinações HOG/LBP/GLCM.

In [ ]:
CLASS_RESULTS = OUT / "classification" / "results"
CLASS_FIGURES = OUT / "classification" / "figures"
!python scripts/run_experiment.py --features "{FEATURES_ALL}" --auto-ablation --outer-splits 5 --inner-splits 3 --seed 42 --output-dir "{CLASS_RESULTS}" --figures-dir "{CLASS_FIGURES}" --plots best

## 4. Sensibilidade ao tratamento de desbalanceamento (opcional, mas já preparado)

Execute se houver tempo para comparar o estudo principal com os mesmos modelos sem `class_weight`.

In [ ]:
NO_WEIGHT_RESULTS = OUT / "classification_no_class_weight" / "results"
NO_WEIGHT_FIGURES = OUT / "classification_no_class_weight" / "figures"
# Descomente para executar:
# !python scripts/run_experiment.py --features "{FEATURES_ALL}" --auto-ablation --class-weight none --outer-splits 5 --inner-splits 3 --seed 42 --output-dir "{NO_WEIGHT_RESULTS}" --figures-dir "{NO_WEIGHT_FIGURES}" --plots none

## 5. Detecção/localização — etapa pesada

Usa regiões candidatas, separação OOF por paciente, undersampling apenas no treino, HistGradientBoosting e NMS.

**Use o `features_regions.csv` regenerado com as 175 regiões da versão atual.**

In [ ]:
DETECTION_RESULTS = OUT / "detection"
!python scripts/run_detection_experiment.py --features-regions "{FEATURES_REGIONS}" --ground-truth "{GROUND_TRUTH}" --image-manifest "{IMAGE_MANIFEST}" --model hist_gradient_boosting --family "HOG+LBP+GLCM" --outer-splits 5 --inner-splits 3 --max-neg-pos-ratio 3 --nms-iou 0.30 --top-k 20 --seed 42 --output-dir "{DETECTION_RESULTS}"

## 6. Conferir os resultados para o artigo

In [ ]:
import json, pandas as pd
print("\n=== CLASSIFICAÇÃO ===")
display(pd.read_csv(CLASS_RESULTS / "summary_formatted.csv"))
print("\n=== DETECÇÃO ===")
print(json.dumps(json.loads((DETECTION_RESULTS / "detection_metrics.json").read_text()), indent=2))
print("\nArquivos finais em:", OUT)

## Saídas preservadas no Drive

- `erik/classification/results/`: métricas, ablação, parâmetros e erros;
- `erik/classification/figures/`: ROC, PR e matriz de confusão;
- `erik/detection/`: OOF regional, predições após NMS, mAP e FROC.

Não use resultados de smoke test no artigo.